In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from utils import *

In [2]:
import os
os.getcwd()

'd:\\GitHub\\smart-crosswalk\\notebooks'

In [11]:
df = pd.read_csv('../data/processed/T2_crosswalk_features.csv')
df.head()

,crosswalk_id,lon,lat,dong_name,admin_dong,lanes,road_rank,max_spd,elderly_ratio,accident_count,crosswalk_length,has_signal,accident_count_50m,accident_count_100m,accident_count_200m,night_accident_ratio,link_count,is_oneway,time_gap_basic
0,10356,127.000192,37.569261,예지동,종로1·2·3·4가동,2.0,104.0,30.0,0.203559,0,21.14050,0,2,4,51,0.000000,NaN,0.0,5.285125
1,10349,126.992498,37.568475,장사동,종로1·2·3·4가동,2.0,104.0,30.0,0.203559,0,21.14050,1,10,21,43,0.666667,4.0,0.0,5.285125
2,136876,126.999299,37.589729,혜화동,혜화동,1.0,104.0,30.0,0.150257,0,10.57025,0,10,16,26,0.312500,3.0,0.0,2.642562
3,136877,126.999124,37.589677,혜화동,혜화동,1.0,104.0,30.0,0.150257,0,10.57025,0,11,16,27,0.312500,3.0,0.0,2.642562
4,136878,126.998376,37.589792,명륜1가,혜화동,1.0,104.0,30.0,0.150257,0,10.57025,0,4,15,25,0.400000,2.0,0.0,2.642562


In [16]:
df.groupby('link_count', as_index=False).agg(count=('link_count', 'count'))

,link_count,count
0,1.0,50
1,2.0,8656
2,3.0,5140
3,4.0,15086
4,5.0,114


In [6]:
df.groupby('is_oneway', as_index=False).agg(count=('is_oneway', 'count'))

,is_oneway,count
0,0.0,30659
1,1.0,234


In [18]:
df = pd.read_csv(CROSSWALK, encoding='cp949')
df.head()

,노드링크 유형,노드 WKT,노드 ID,노드 유형 코드,링크 WKT,링크 ID,링크 유형 코드,시작노드 ID,종료노드 ID,링크 길이,시군구코드,시군구명,읍면동코드,읍면동명
0,LINK,NaN,0,NaN,LINESTRING(126.9736912264903 37.56998557196629...,132084,1000.0,132492.0,72457.0,38.404,1111000000,종로구,1111012000,신문로1가
1,NODE,POINT(127.00019246086138 37.56926064851388),10356,0.0,NaN,0,NaN,NaN,NaN,NaN,1111000000,종로구,1111015800,예지동
2,NODE,POINT(126.99249846867598 37.568475047964434),10349,0.0,NaN,0,NaN,NaN,NaN,NaN,1111000000,종로구,1111015400,장사동
3,NODE,POINT(126.99929873885362 37.589729124195195),136876,0.0,NaN,0,NaN,NaN,NaN,NaN,1111000000,종로구,1111016900,혜화동
4,NODE,POINT(126.9991242525229 37.58967734906277),136877,0.0,NaN,0,NaN,NaN,NaN,NaN,1111000000,종로구,1111016900,혜화동


In [ ]:
df = pd.read_csv(PED_SIGNAL, encoding='cp949')
df.groupby('자치구', as_index=False).agg(계=('자치구', 'count'))
df.head()

In [ ]:
import os
import re

folder = '../data/raw/traffic_volume'  # 실제 경로로 수정

for fname in os.listdir(folder):
    if not fname.endswith('.xlsx'):
        continue
    
    # "01월 서울시 교통량 조사자료(2025).xlsx" → month="01", year="2025"
    match = re.match(r'(\d{2})월\s.*\((\d{4})\)\.xlsx', fname)
    if match:
        month = match.group(1)
        year = match.group(2)
        new_name = f'{year}_{month}.xlsx'
        
        old_path = os.path.join(folder, fname)
        new_path = os.path.join(folder, new_name)
        
        print(f'{fname}  →  {new_name}')
        os.rename(old_path, new_path)

print('\n완료!')

In [ ]:
# notebooks/01_data_check.ipynb

import sys
sys.path.insert(0, '..')

from utils import *
import pandas as pd
import geopandas as gpd

# === 1. 사고 데이터 ===
acc = pd.read_csv(ACCIDENTS)
print("=== 사고 데이터 ===")
print(f"행: {len(acc)}, 열: {len(acc.columns)}")
print(acc.columns.tolist())
print(acc[['lon','lat']].describe())
print(acc.head(3))
print()

# === 2. 횡단보도 ===
cw = pd.read_csv(CROSSWALK, encoding='cp949')
print("=== 횡단보도 ===")
print(f"행: {len(cw)}, 열: {len(cw.columns)}")
print(cw.columns.tolist())
print(cw.head(3))
print()

# === 3. 보행자신호등 ===
ped = pd.read_csv(PED_SIGNAL, encoding='cp949')
print("=== 보행자신호등 ===")
print(f"행: {len(ped)}, 열: {len(ped.columns)}")
print(ped.columns.tolist())
print(ped.head(3))
print()

# === 4. 고령인구 === 
eld = pd.read_csv(ELDERLY_POP_DONG, encoding='utf-8-sig', header=None, nrows=10)
print("=== 고령인구 (동별) ===")
print(eld.to_string())
print()

# 실제 데이터 (헤더 4줄 스킵)
eld = pd.read_csv(ELDERLY_POP_DONG, encoding='utf-8-sig', skiprows=4)
eld.columns = ['시도', '구', '동', '전체_소계', '전체_남', '전체_여',
               '노인_소계', '노인_남', '노인_여',
               '노인_내국_소계', '노인_내국_남', '노인_내국_여',
               '노인_외국_소계', '노인_외국_남', '노인_외국_여']
print(f"행: {len(eld)}, 열: {len(eld.columns)}")
print(eld.head(10))

# === 5. 표준노드링크 ===
link = gpd.read_file(MOCT_LINK_SHP, encoding='cp949')
print("=== 표준노드링크 (LINK) ===")
print(f"행: {len(link)}, 열: {len(link.columns)}")
print(link.columns.tolist())
print(link.head(3))
print()

# === 6. 교통량 (1월만 샘플) ===
tv = pd.read_excel(TRAFFIC_DIR / "2025_01.xlsx")
print("=== 교통량 ===")
print(f"행: {len(tv)}, 열: {len(tv.columns)}")
print(tv.columns.tolist())
print(tv.head(3))

In [ ]:
# 사고 유형 확인 (이게 진짜 사고 유형 컬럼)
print("=== acdnt_dc 분포 ===")
print(acc['acdnt_dc'].value_counts())
print()

# 올바른 필터
elderly_cross = acc[
    (acc['dmge_age_2'] == '65세 이상') &
    (acc['acdnt_dc'].str.contains('횡단', na=False))
]
print(f"65세↑ + 횡단중: {len(elderly_cross)}건 / 전체 {len(acc)}건")
print()

# 65세 이상 전체 (횡단 아닌 것 포함)
elderly_all = acc[acc['dmge_age_2'] == '65세 이상']
print(f"65세↑ 전체: {len(elderly_all)}건")
print(elderly_all['acdnt_dc'].value_counts())

In [ ]:
import pandas as pd
import os

# === 1. 지점 좌표 테이블 (시트 2, 아무 월이나 하나에서) ===
stations = pd.read_excel(
    TRAFFIC_DIR / "2025_01.xlsx", 
    sheet_name=2
)
print("=== 교통량 지점 ===")
print(f"지점 수: {len(stations)}")
print(stations.columns.tolist())
print(stations.head(3))
print()

# 중구 근처 지점 확인 (위도 37.55~37.57, 경도 126.97~127.01 정도)
stations_junggu = stations[
    (stations['위도'] > 37.54) & (stations['위도'] < 37.58) &
    (stations['경도'] > 126.96) & (stations['경도'] < 127.02)
]
print(f"중구 근처 지점: {len(stations_junggu)}개")
print(stations_junggu[['지점번호', '지점명칭', '위도', '경도']])
print()

# === 2. 12개월 교통량 합치기 ===
all_traffic = []
for fname in sorted(os.listdir(TRAFFIC_DIR)):
    if not fname.endswith('.xlsx'):
        continue
    df = pd.read_excel(TRAFFIC_DIR / fname, sheet_name=1)
    all_traffic.append(df)
    print(f"  {fname}: {len(df)}행")

traffic = pd.concat(all_traffic, ignore_index=True)
print(f"\n전체 교통량: {len(traffic)}행")
print(traffic.columns.tolist())
print()

# === 3. 지점별 일평균 교통량 (AADT) ===
# 시간대 컬럼 합산 → 일 교통량
hour_cols = [f'{h}시' for h in range(24)]
traffic['일교통량'] = traffic[hour_cols].apply(
    lambda row: pd.to_numeric(row, errors='coerce').sum(), axis=1
)

# 지점별 연평균
aadt = traffic.groupby('지점번호')['일교통량'].mean().reset_index()
aadt.columns = ['지점번호', 'AADT']
print("=== 지점별 AADT ===")
print(aadt.head(10))